# RQ1 — Fine-tuning **LightGBM** (dự báo kWh / phiên sạc)

**RQ1:** *Yếu tố nào (giờ trong ngày, loại địa điểm, thứ trong tuần) ảnh hưởng mạnh nhất đến điện năng tiêu thụ mỗi phiên sạc EV?*

Notebook này **chỉ làm fine-tuning siêu tham số** cho LightGBM. Nó **không** thay thế notebook RQ1 gốc:
cell 0–3 được sao y hệt bản gốc (cùng 9 feature, cùng 5 nhóm yếu tố, cùng mốc chia tập,
cùng `reg_metrics` / `cv_rmse` / `run_benchmark`) để kết quả đặt cạnh nhau được.

### Giao thức fine-tuning (GIỐNG HỆT ở cả 3 notebook RQ1 — chỉ không gian tìm kiếm khác)

| Hạng mục | Quy ước |
|---|---|
| **Thư viện tuning** | **Optuna** (TPE, `multivariate=True, group=True`) — phù hợp nhất ở đây: không gian có cả int/float/log/categorical và có tham số điều kiện, TPE lại hiệu quả khi ngân sách trial nhỏ |
| **Thước đo chọn** | **CV RMSE (kWh)** trung bình trên **đúng 5 fold expanding-window** của cell 2 |
| **Chia tập** | Train → hết `2019-10-31` · Val → hết `2020-01-15` · Test `2020-01-16` → hết `2021-09-13` |
| **Giai đoạn 1** | Dò rộng toàn không gian, `N_TRIALS_S1` trial |
| **Giai đoạn 2** | Co mỗi khoảng về ±35% quanh best của giai đoạn 1, `N_TRIALS_S2` trial |
| **Chốt HP** | Chạy lại **top-{K}** cấu hình với **nhiều seed**, chọn cấu hình có CV RMSE trung bình qua seed tốt nhất (tránh chọn đúng cái may mắn thấp nhất một lần) |
| **Test** | **Không** tham gia bất kỳ bước chọn nào; chỉ dùng một lần ở cell cuối |
| **Đầu ra** | `{tag}_best_hyperparameters.json`, `{tag}_tuning_trials.csv`, `{tag}_stability_check.csv`, `{tag}_cv_folds_tuned.csv`, `{tag}_test_tuned.csv`, `{tag}_test_predictions.csv` + 3 biểu đồ PNG |

**Chạy trên Kaggle:** Add Data → dataset ACN (`acn_data(cleaned).xlsx`) · Accelerator **None (CPU)** · Internet **On** (để `pip install optuna`) · Run All.

> **Vì sao Optuna cho LightGBM:** 13 siêu tham số, trong đó `num_leaves` ↔ `max_depth` ràng buộc nhau
> và 3 tham số riêng cho cột category (`cat_smooth`, `cat_l2`, `max_cat_threshold`) — `stationID` có 115 mức
> nên nhóm này ảnh hưởng thật. Grid search sẽ nổ tổ hợp; TPE multivariate học được tương quan giữa các tham số.

## 0. Cấu hình chung + đọc dữ liệu + chia tập (sao y notebook RQ1 gốc, cell 0–3)

In [ ]:

# ===== CELL 0: Import + cấu hình CHUNG (GIỮ NGUYÊN giao thức của notebook RQ1 gốc) =====
# Notebook này CHỈ làm fine-tuning. Cell 0-3 được sao lại y hệt notebook RQ1 gốc để
# 3 model (LightGBM / XGBoost / Linear Regression) nằm trên ĐÚNG CÙNG một benchmark:
# cùng 9 FEATURES, cùng 5 nhóm yếu tố, cùng mốc chia tập, cùng 5 fold, cùng hàm metric.
import os, glob, json, time, warnings, platform, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')

SEED   = 42
TARGET = 'kWhDelivered'                          # kWh / phiên sạc, GIỮ NGUYÊN thang gốc (không log) cho cả 3 model
FEATURES = ['siteID', 'stationID', 'hour_sin', 'hour_cos', 'day_of_week', 'is_weekend', 'is_holiday', 'month', 'seasonal']
CAT_COLS = ['siteID', 'stationID', 'day_of_week', 'month', 'seasonal']
# RQ1 xếp hạng theo NHÓM yếu tố (mỗi nhóm bị xáo CÙNG MỘT hoán vị); cặp cột trùng thông tin phải đi chung nhóm.
FACTOR_GROUPS = {'hour': ['hour_sin', 'hour_cos'],
                 'day_of_week': ['day_of_week', 'is_weekend'],
                 'location': ['siteID', 'stationID'],
                 'month': ['month', 'seasonal'],
                 'is_holiday': ['is_holiday']}
FACTOR_NAME = {'hour': 'Time of day (hour_sin + hour_cos)',
               'day_of_week': 'Day of week (+ is_weekend)',
               'location': 'Location (siteID + stationID)',
               'month': 'Month (+ seasonal)',
               'is_holiday': 'Holiday (is_holiday)'}
GROUP_KEYS = list(FACTOR_GROUPS)
assert sorted(sum(FACTOR_GROUPS.values(), [])) == sorted(FEATURES), 'mỗi feature phải thuộc đúng 1 nhóm'

# --- Mốc thời gian (GIỮ NGUYÊN notebook gốc; cả 3 mốc đều tính HẾT ngày) ---
TRAIN_END   = '2019-10-31'   # Train = đầu dữ liệu -> hết 2019-10-31
VAL_END     = '2020-01-15'   # Val   = 2019-11-01  -> hết 2020-01-15
TEST_END    = '2021-09-13'   # Test  = 2020-01-16  -> hết 2021-09-13 (cuối tập dữ liệu)

# --- Time-based CV trong vùng Train+Val (expanding window) ---
N_FOLDS     = 5              # 5 fold
FOLD_MONTHS = 2              # mỗi cửa sổ validation dài 2 tháng, lùi dần từ VAL_END
GAP_MIN_DAYS = 5
N_PERM_REPEATS = 20

# --- Ngân sách fine-tuning (chỉ cell này là phần MỚI so với notebook gốc) ---
FAST_MODE      = False       # True = smoke test, chỉ để kiểm tra code chạy được
N_TRIALS_S1    = 60          # giai đoạn 1: dò rộng toàn không gian
N_TRIALS_S2    = 40          # giai đoạn 2: dò hẹp quanh vùng tốt của giai đoạn 1
TOP_K_STABLE   = 5           # số cấu hình tốt nhất đem đi kiểm tra độ ổn định
STABLE_SEEDS   = [42, 202, 1337]      # seed dùng cho kiểm tra độ ổn định
if FAST_MODE:
    N_TRIALS_S1, N_TRIALS_S2, TOP_K_STABLE, STABLE_SEEDS = 6, 4, 2, [42]

# --- Tìm file dữ liệu: ưu tiên dataset Kaggle đã attach, chấp nhận mọi tên acn*.xlsx ---
def resolve_data():
    pats = ['/kaggle/input/**/acn_data(cleaned) - (RQ1).xlsx', '/kaggle/input/**/acn_data(cleaned).xlsx',
            '/kaggle/input/**/acn*.xlsx', '/mnt/project/acn*.xlsx', './acn*.xlsx', '../input/**/acn*.xlsx']
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError('Không tìm thấy file acn*.xlsx. Hãy Add Data dataset ACN vào notebook.')
DATA_PATH = resolve_data()
print('Data        :', DATA_PATH)
print('Python      :', sys.version.split()[0], '|', platform.platform())
print('Ngân sách   : stage1 =', N_TRIALS_S1, 'trial | stage2 =', N_TRIALS_S2,
      'trial | top-K ổn định =', TOP_K_STABLE, 'x', len(STABLE_SEEDS), 'seed')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './outputs_rq1_tuning'
os.makedirs(OUT_DIR, exist_ok=True)
print('Thư mục ra  :', OUT_DIR)

In [ ]:

# ===== CELL 1: Đọc dữ liệu + LIỆT KÊ TẤT CẢ CỘT + kiểm tra (y hệt notebook gốc) =====
try:
    df = pd.read_excel(DATA_PATH, sheet_name='full_data')
except ValueError:
    df = pd.read_excel(DATA_PATH)                 # file chỉ có 1 sheet, tên khác
df = df.sort_values('connectionTime', kind='stable').reset_index(drop=True)
if 'hour' not in df.columns:
    _ct = df['connectionTime']; df['hour'] = _ct.dt.hour + _ct.dt.minute / 60 + _ct.dt.second / 3600
    print('Cột hour chưa có trong file Excel -> đã tính từ connectionTime')
# Kiểu category bị mất sau khi qua Excel -> ép lại với bảng nhãn CỐ ĐỊNH (chỉ là danh sách nhãn, không chứa target)
CAT_LEVELS = {'day_of_week': list(range(7)), 'month': list(range(1, 13)), 'seasonal': list(range(4)),
              'siteID': sorted(df['siteID'].unique()), 'stationID': sorted(df['stationID'].unique())}
assert set(df['seasonal'].unique()) <= set(CAT_LEVELS['seasonal']), 'seasonal ngoài 0-3 -> sẽ bị NaN khi ép category'
for c in CAT_COLS:
    df[c] = pd.Categorical(df[c], categories=CAT_LEVELS[c])
df['hour_bin'] = np.floor(df['hour']).astype(int)      # CHỈ để chẩn đoán, KHÔNG đưa vào model
print('Shape:', df.shape)

MEDIATOR = ['disconnectTime', 'doneChargingTime', 'charging_duration', 'parking_duration']
LAG_ROLL = ['lag_1_log', 'lag_3_log', 'lag_5_log', 'rolling_mean_3_log', 'rolling_mean_5_log']
LEAK = MEDIATOR + LAG_ROLL

# --- kiểm tra logic (giống notebook gốc) ---
assert set(FEATURES + [TARGET, 'connectionTime']) <= set(df.columns)
assert not any(c in FEATURES for c in LEAK), 'FEATURES đang chứa cột rò rỉ'
assert not any(c.startswith(('lag_', 'rolling_', 'st_', 'roll')) for c in FEATURES), 'RQ1 không dùng lag/rolling'
assert df[FEATURES + [TARGET]].isna().sum().sum() == 0
assert (df[TARGET] > 0).all(), 'MAPE cần y > 0'
assert np.floor(df.hour).eq(df.connectionTime.dt.hour).all() and df.connectionTime.dt.dayofweek.eq(df.day_of_week.astype(int)).all()
print('OK. Số phiên theo site:'); print(df.siteID.value_counts().sort_index().to_string())
print('kWh: mean %.3f | median %.3f | std %.3f | min %.3f | max %.3f'
      % (df[TARGET].mean(), df[TARGET].median(), df[TARGET].std(), df[TARGET].min(), df[TARGET].max()))

In [ ]:

# ===== CELL 2: Time-Based Split (Train / Val / Test + 5 fold expanding window) — y hệt notebook gốc =====
t = df['connectionTime']
TRAIN_END_X = pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1)     # mốc LOẠI TRỪ (nửa mở)
VAL_END_X   = pd.Timestamp(VAL_END)   + pd.Timedelta(days=1)
TEST_END_X  = pd.Timestamp(TEST_END)  + pd.Timedelta(days=1)
assert TRAIN_END_X < VAL_END_X < TEST_END_X

df['set'] = np.select([t < TRAIN_END_X, t < VAL_END_X, t < TEST_END_X], ['Train', 'Val', 'Test'], 'Excluded')
N_EXCL = int((df['set'] == 'Excluded').sum())
DEV  = np.flatnonzero(df['set'].isin(['Train', 'Val']).values)
TEST = np.flatnonzero((df['set'] == 'Test').values)
assert len(DEV) + len(TEST) + N_EXCL == len(df)
assert df.loc[DEV, 'connectionTime'].max() < df.loc[TEST, 'connectionTime'].min()

edges = pd.DatetimeIndex([VAL_END_X - pd.DateOffset(months=FOLD_MONTHS * i) for i in range(N_FOLDS, -1, -1)])
assert edges[-1] == VAL_END_X and edges[0] > t.min()
FOLDS, FOLD_INFO = [], []
for k in range(N_FOLDS):
    a, b = edges[k], edges[k + 1]
    tr = np.flatnonzero((t < a).values); va = np.flatnonzero(((t >= a) & (t < b)).values)
    assert len(tr) > 0 and len(va) > 0 and t.iloc[tr].max() < t.iloc[va].min()
    FOLDS.append((tr, va))
    FOLD_INFO.append({'fold': k + 1, 'train': f'{t.iloc[0].date()} -> {(a - pd.Timedelta(days=1)).date()}',
                      'val': f'{a.date()} -> {(b - pd.Timedelta(days=1)).date()}', 'n_train': len(tr), 'n_val': len(va)})
FOLD_INFO = pd.DataFrame(FOLD_INFO).set_index('fold')
print('Time-based CV (expanding window) — 5 fold dùng để CHỌN SIÊU THAM SỐ:'); print(FOLD_INFO.to_string())

sets = df.groupby('set').agg(n_sessions=(TARGET, 'size'), start=('connectionTime', 'min'), end=('connectionTime', 'max'))
print('\n', sets.loc[['Train', 'Val', 'Test']].to_string())
print(f'\nPhiên sau TEST_END bị loại: {N_EXCL}')

In [ ]:

# ===== CELL 3: Metric chung + động cơ benchmark chung (y hệt notebook gốc) =====
X_ALL = df[FEATURES].copy()
for c in CAT_COLS:
    X_ALL[c] = pd.Categorical(df[c], categories=CAT_LEVELS[c])
Y_ALL = df[TARGET].to_numpy(float)

def reg_metrics(y, p):
    # QUY ƯỚC DẤU: sai số = thực tế - dự đoán
    y, p = np.asarray(y, float), np.asarray(p, float)
    mse = mean_squared_error(y, p)
    return {'MAE': mean_absolute_error(y, p), 'MSE': mse, 'RMSE': np.sqrt(mse), 'R2': r2_score(y, p),
            'ME': np.mean(y - p),
            'MAPE_tham_khao_%': 100 * np.mean(np.abs((y - p) / y)),
            'WAPE_%': 100 * np.abs(y - p).sum() / np.abs(y).sum()}
METRICS = ['MAE', 'MSE', 'RMSE', 'R2', 'ME', 'WAPE_%']   # MAPE chỉ tham khảo, cố ý không nằm ở đây

def predict_kwh(model, X):                      # kWh không âm (LR có thể ra số âm)
    return np.clip(model.predict(X), 0, None)

def _default_X():
    # Model nào cần mã hoá riêng (ví dụ Linear Regression) sẽ đặt X_FIT ở cell 5;
    # tra cứu MUỘN ở đây để mọi cell chẩn đoán dùng đúng ma trận đặc trưng của model đó.
    return globals().get('X_FIT', X_ALL)

def cv_rmse(build_and_fit, hp, X=None):         # THƯỚC ĐO CHỌN SIÊU THAM SỐ: RMSE trung bình trên ĐÚNG 5 fold
    X = _default_X() if X is None else X
    s = []
    for tr, va in FOLDS:
        m = build_and_fit(X.iloc[tr], Y_ALL[tr], hp)
        s.append(np.sqrt(mean_squared_error(Y_ALL[va], predict_kwh(m, X.iloc[va]))))
    return float(np.mean(s))

def cv_rmse_folds(build_and_fit, hp, X=None):   # MỚI: trả về RMSE của TỪNG fold (để xem độ ổn định)
    X = _default_X() if X is None else X
    s = []
    for tr, va in FOLDS:
        m = build_and_fit(X.iloc[tr], Y_ALL[tr], hp)
        s.append(np.sqrt(mean_squared_error(Y_ALL[va], predict_kwh(m, X.iloc[va]))))
    return np.array(s, float)

def run_benchmark(name, build_and_fit, hp, X=None):
    X = _default_X() if X is None else X
    rows, oof = [], np.full(len(df), np.nan)
    fold_models = []
    for k, (tr, va) in enumerate(FOLDS, 1):
        m = build_and_fit(X.iloc[tr], Y_ALL[tr], hp)
        fold_models.append((m, va))
        p = predict_kwh(m, X.iloc[va]); oof[va] = p
        rows.append({'fold': k, 'val': FOLD_INFO.loc[k, 'val'], **reg_metrics(Y_ALL[va], p)})
    cv = pd.DataFrame(rows, index=range(1, len(FOLDS) + 1))
    final = build_and_fit(X.iloc[DEV], Y_ALL[DEV], hp)                  # refit trên Train+Val -> chấm Test
    p_te = predict_kwh(final, X.iloc[TEST]); y_te = Y_ALL[TEST]
    test_all = pd.Series(reg_metrics(y_te, p_te), name=name)
    pred = df.loc[TEST, ['connectionTime', 'siteID', 'stationID', 'hour', 'hour_bin', 'day_of_week', 'month']].assign(
        actual=y_te, pred=p_te, residual=y_te - p_te)
    return dict(name=name, cv=cv, test_all=test_all, pred=pred, model=final, oof=oof, fold_models=fold_models)

def summarize(res):
    cv = res['cv'][METRICS]
    print(f"=== {res['name']} | CV {N_FOLDS} fold (mean ± std) ===")
    print(pd.DataFrame({'mean': cv.mean(), 'std': cv.std()}).round(3).T.to_string())
    print(f"\n=== {res['name']} | TEST toàn bộ ({len(TEST)} phiên) ===")
    print(res['test_all'].round(3).to_frame().T.to_string())

print('Hàm benchmark chung đã sẵn sàng. Thước đo chọn siêu tham số = CV RMSE (kWh) trên 5 fold.')

## 1. Khung fine-tuning 2 giai đoạn (dùng chung cho cả 3 notebook RQ1)

In [ ]:

# ===== CELL 4: Khung fine-tuning DÙNG CHUNG cho cả 3 notebook RQ1 (chỉ không gian tìm kiếm là khác) =====
# Thiết kế 2 giai đoạn:
#   Giai đoạn 1 (dò rộng): TPE quét toàn bộ không gian -> tìm vùng tốt.
#   Giai đoạn 2 (dò hẹp) : co mỗi khoảng về quanh giá trị tốt nhất của giai đoạn 1 -> tinh chỉnh.
# Thước đo duy nhất để chọn: CV RMSE (kWh) trên ĐÚNG 5 fold expanding-window của cell 2.
# Test KHÔNG được dùng ở bất kỳ bước chọn nào.
import subprocess
try:
    import optuna
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'], check=True)
    import optuna
from optuna.importance import get_param_importances
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Optuna', optuna.__version__)

# ---------- Không gian tìm kiếm khai báo bằng dữ liệu (để co khoảng ở giai đoạn 2 một cách tự động) ----------
# ('int',   lo, hi, step_or_None, log_bool)
# ('float', lo, hi, log_bool)
# ('cat',   [các lựa chọn])
def suggest(trial, name, spec):
    k = spec[0]
    if k == 'int':
        _, lo, hi, step, log = spec
        lo, hi = int(lo), int(max(hi, lo))
        if log:   return trial.suggest_int(name, lo, hi, log=True)
        if step:  return trial.suggest_int(name, lo, hi, step=max(1, int(step)))
        return trial.suggest_int(name, lo, hi)
    if k == 'float':
        _, lo, hi, log = spec
        if hi <= lo: hi = lo * 1.000001 + 1e-12
        return trial.suggest_float(name, lo, hi, log=log)
    if k == 'cat':
        return trial.suggest_categorical(name, spec[1])
    raise ValueError(spec)

def narrow(spec, best, shrink=0.35, cats_keep=2, ordered_cat=None):
    """Co khoảng về quanh `best`. shrink = nửa độ rộng mới tính theo % độ rộng cũ."""
    k = spec[0]
    if k == 'int':
        _, lo, hi, step, log = spec
        if log:
            f = (hi / max(lo, 1e-9)) ** shrink
            nlo, nhi = max(lo, int(np.floor(best / f))), min(hi, int(np.ceil(best * f)))
        else:
            half = max(1, int(round((hi - lo) * shrink)))
            nlo, nhi = max(lo, best - half), min(hi, best + half)
        if step: nlo, nhi = max(lo, nlo - (nlo - lo) % step), nhi
        return ('int', nlo, max(nhi, nlo), step, log)
    if k == 'float':
        _, lo, hi, log = spec
        if log:
            f = (hi / max(lo, 1e-12)) ** shrink
            return ('float', max(lo, best / f), min(hi, best * f), True)
        half = (hi - lo) * shrink
        return ('float', max(lo, best - half), min(hi, best + half), False)
    if k == 'cat':
        ch = list(spec[1])
        if len(ch) <= cats_keep: return spec
        if best in ch:
            i = ch.index(best); keep = ch[max(0, i - 1): i + 2] if ordered_cat else [best]
            return ('cat', keep if keep else [best])
        return spec
    raise ValueError(spec)

def narrow_space(space, best_params, shrink=0.35, ordered=()):
    return {k: (narrow(v, best_params[k], shrink, ordered_cat=(k in ordered)) if k in best_params else v)
            for k, v in space.items()}

def space_repr(space):
    rows = []
    for k, v in space.items():
        if v[0] == 'cat': rows.append({'hyperparameter': k, 'kiểu': 'categorical', 'khoảng / lựa chọn': str(v[1])})
        elif v[0] == 'int': rows.append({'hyperparameter': k, 'kiểu': 'int' + (' (log)' if v[4] else ''),
                                         'khoảng / lựa chọn': f'[{v[1]}, {v[2]}]' + (f' step {v[3]}' if v[3] else '')})
        else: rows.append({'hyperparameter': k, 'kiểu': 'float' + (' (log)' if v[3] else ''),
                           'khoảng / lựa chọn': f'[{v[1]:.6g}, {v[2]:.6g}]'})
    return pd.DataFrame(rows).set_index('hyperparameter')

# ---------- Động cơ chạy 1 study ----------
def run_study(space, n_trials, tag, seed=SEED, fixed=None, X=None):
    fixed = fixed or {}
    def objective(trial):
        hp = {k: suggest(trial, k, v) for k, v in space.items()}
        hp = post_process_hp({**hp, **fixed})            # hàm riêng của từng model (cell 5)
        t0 = time.time()
        v = cv_rmse(build_and_fit, hp, X=X)
        trial.set_user_attr('fit_seconds', round(time.time() - t0, 2))
        return v
    st = optuna.create_study(direction='minimize', study_name=tag,
                             sampler=optuna.samplers.TPESampler(seed=seed, multivariate=True, group=True))
    t0 = time.time()
    st.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f'[{tag}] {n_trials} trial trong {(time.time()-t0)/60:.1f} phút | best CV RMSE = {st.best_value:.4f} kWh')
    return st

def trials_df(st, stage):
    rows = []
    for tr in st.trials:
        if tr.value is None: continue
        rows.append({'stage': stage, 'trial': tr.number, 'cv_rmse': tr.value,
                     'fit_seconds': tr.user_attrs.get('fit_seconds', np.nan), **tr.params})
    return pd.DataFrame(rows)
print('Khung fine-tuning đã sẵn sàng.')

## 2. Định nghĩa LightGBM và không gian tìm kiếm

In [ ]:

# ===== CELL 5 (LightGBM): định nghĩa model + không gian tìm kiếm =====
import lightgbm as lgb
print('LightGBM', lgb.__version__)
MODEL_NAME, MODEL_TAG = 'LightGBM', 'lightgbm'
X_FIT = X_ALL                                   # LightGBM đọc trực tiếp dtype 'category' -> KHÔNG one-hot

# Siêu tham số CỐ ĐỊNH (không tune): đảm bảo tái lập và không dùng val để early-stop
FIXED_HP = dict(objective='regression', random_state=SEED, n_jobs=-1, deterministic=True,
                force_row_wise=True, verbosity=-1, subsample_freq=1)

def seed_override(sd):                          # dùng ở cell 9 khi kiểm tra ổn định đa seed
    return {'random_state': sd}

def post_process_hp(hp):
    hp = dict(hp)
    # subsample < 1 chỉ có tác dụng khi subsample_freq >= 1
    hp.setdefault('subsample_freq', 1)
    # num_leaves không được vượt 2^max_depth (nếu vượt thì max_depth là ràng buộc thật sự)
    if 'num_leaves' in hp and 'max_depth' in hp:
        hp['num_leaves'] = int(min(hp['num_leaves'], max(2, 2 ** int(hp['max_depth']) - 1)))
    return hp

def cast_params(d):                             # đọc lại từ CSV/DataFrame -> ép đúng kiểu
    ints = {'n_estimators', 'num_leaves', 'max_depth', 'min_child_samples', 'subsample_freq'}
    return {k: (int(v) if k in ints else v) for k, v in d.items()}

def build_and_fit(X, y, hp):
    return lgb.LGBMRegressor(**hp).fit(X, y)

# Số cây LÀ siêu tham số (không early-stopping -> không dùng val để dừng, giữ đúng thiết kế notebook gốc)
SPACE1 = {
    'n_estimators':     ('int',   100, 2000, 50, False),
    'learning_rate':    ('float', 0.005, 0.3, True),
    'num_leaves':       ('int',   4, 96, None, False),
    'max_depth':        ('int',   3, 12, None, False),
    'min_child_samples':('int',   10, 400, None, True),
    'subsample':        ('float', 0.5, 1.0, False),
    'colsample_bytree': ('float', 0.5, 1.0, False),
    'reg_alpha':        ('float', 1e-8, 20.0, True),
    'reg_lambda':       ('float', 1e-8, 20.0, True),
    'min_split_gain':   ('float', 1e-8, 1.0, True),
    'cat_smooth':       ('float', 1.0, 100.0, True),      # LightGBM: làm trơn thống kê của cột category
    'cat_l2':           ('float', 1e-3, 50.0, True),
    'max_cat_threshold':('int',   8, 64, None, False),
}
ORDERED_CATS = ()                               # không có categorical có thứ tự trong không gian này
print(f'{MODEL_NAME}: {len(SPACE1)} siêu tham số được tune')

## 3. ⭐ Giai đoạn 1 — dò rộng

In [ ]:

# ===== CELL 6: GIAI ĐOẠN 1 — dò rộng toàn không gian =====
print('Không gian tìm kiếm GIAI ĐOẠN 1:'); print(space_repr(SPACE1).to_string())
print()
ST1 = run_study(SPACE1, N_TRIALS_S1, f'{MODEL_TAG}_stage1')
TR1 = trials_df(ST1, 'stage1')
print('\nTop-5 giai đoạn 1:')
print(TR1.nsmallest(5, 'cv_rmse').set_index('trial').round(5).to_string())

## 4. ⭐ Giai đoạn 2 — dò hẹp quanh vùng tốt

In [ ]:

# ===== CELL 7: GIAI ĐOẠN 2 — co khoảng quanh vùng tốt nhất rồi tinh chỉnh =====
SPACE2 = narrow_space(SPACE1, ST1.best_params, shrink=0.35, ordered=ORDERED_CATS)
print('Không gian tìm kiếm GIAI ĐOẠN 2 (đã co quanh best của giai đoạn 1):')
print(space_repr(SPACE2).to_string()); print()
ST2 = run_study(SPACE2, N_TRIALS_S2, f'{MODEL_TAG}_stage2', seed=SEED + 1)
TR2 = trials_df(ST2, 'stage2')

TRIALS = pd.concat([TR1, TR2], ignore_index=True)
TRIALS['rank'] = TRIALS['cv_rmse'].rank(method='min').astype(int)
TRIALS = TRIALS.sort_values('cv_rmse').reset_index(drop=True)
TRIALS.to_csv(f'{OUT_DIR}/{MODEL_TAG}_tuning_trials.csv', index=False)
print(f'\nGộp 2 giai đoạn: {len(TRIALS)} trial hợp lệ. Đã lưu {MODEL_TAG}_tuning_trials.csv')
print('\n10 trial tốt nhất (toàn bộ):')
print(TRIALS.head(10).drop(columns=['rank']).set_index(['stage', 'trial']).round(5).to_string())
better = ST2.best_value < ST1.best_value
print(f'\nGiai đoạn 2 {"CẢI THIỆN" if better else "KHÔNG cải thiện"}: '
      f'{ST1.best_value:.4f} -> {ST2.best_value:.4f} kWh '
      f'({100*(ST1.best_value-ST2.best_value)/ST1.best_value:+.2f}%)')

## 5. Biểu đồ quá trình tuning & độ quan trọng siêu tham số

In [ ]:

# ===== CELL 8: Biểu đồ quá trình fine-tuning + độ quan trọng của từng siêu tham số =====
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
# (a) lịch sử tối ưu của 2 giai đoạn
for st, lab, c in [(ST1, 'Stage 1 (broad)', '#4C78A8'), (ST2, 'Stage 2 (refined)', '#F58518')]:
    v = np.array([t.value for t in st.trials if t.value is not None])
    ax[0].plot(range(1, len(v) + 1), v, 'o', ms=3.5, alpha=.45, color=c)
    ax[0].plot(range(1, len(v) + 1), np.minimum.accumulate(v), '-', lw=1.8, color=c, label=lab)
ax[0].set(xlabel='Trial', ylabel='CV RMSE (kWh)', title=f'{MODEL_NAME}: Optuna optimisation history')
ax[0].legend(fontsize=8)

# (b) độ quan trọng của siêu tham số (fANOVA trên toàn bộ trial của giai đoạn 1 = không gian đầy đủ)
try:
    imp = pd.Series(get_param_importances(ST1)).sort_values()
    ax[1].barh(imp.index, imp.values, color='#54A24B')
    ax[1].set(xlabel='Importance (fANOVA)', title=f'{MODEL_NAME}: hyperparameter importance')
except Exception as e:
    ax[1].text(.5, .5, f'không tính được\n{e}', ha='center', va='center'); imp = pd.Series(dtype=float)

# (c) phân phối CV RMSE của mọi trial: cho thấy fine-tuning thu được bao nhiêu
ax[2].hist(TRIALS['cv_rmse'], bins=25, color='#B279A2', edgecolor='white')
ax[2].axvline(TRIALS['cv_rmse'].min(), color='r', ls='--', lw=1.5, label=f'best = {TRIALS["cv_rmse"].min():.4f}')
ax[2].axvline(TRIALS['cv_rmse'].median(), color='k', ls=':', lw=1.5, label=f'median = {TRIALS["cv_rmse"].median():.4f}')
ax[2].set(xlabel='CV RMSE (kWh)', ylabel='Số trial', title=f'{MODEL_NAME}: spread of all trials'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{MODEL_TAG}_tuning_overview.png', dpi=130, bbox_inches='tight'); plt.show()

if len(imp):
    print('Độ quan trọng siêu tham số (fANOVA, giai đoạn 1):')
    print(imp.sort_values(ascending=False).round(4).to_string())

## 6. ⭐ Kiểm tra độ ổn định đa seed và CHỐT siêu tham số

In [ ]:

# ===== CELL 9: Kiểm tra ĐỘ ỔN ĐỊNH của top-K rồi CHỐT siêu tham số cuối =====
# Lý do: CV RMSE tốt nhất là giá trị NHỎ NHẤT trong hàng chục lượt thử nên nó lạc quan.
# Ta chạy lại top-K cấu hình với nhiều seed khác nhau và chọn cấu hình có
# (trung bình qua seed) tốt nhất — ổn định hơn là chọn đúng cái thấp nhất một lần.
PARAM_COLS = [c for c in TRIALS.columns if c not in ('stage', 'trial', 'cv_rmse', 'fit_seconds', 'rank')]
cands = TRIALS.head(TOP_K_STABLE)
rows = []
for _, r in cands.iterrows():
    hp_raw = {c: r[c] for c in PARAM_COLS if pd.notna(r[c])}
    hp_raw = cast_params(hp_raw)
    for sd in STABLE_SEEDS:
        hp = post_process_hp({**hp_raw, **FIXED_HP, **seed_override(sd)})
        f = cv_rmse_folds(build_and_fit, hp)
        rows.append({'cấu hình': f"{r['stage']}#{int(r['trial'])}", 'seed': sd,
                     'cv_rmse_mean': f.mean(), 'cv_rmse_std': f.std(),
                     **{f'fold{i+1}': v for i, v in enumerate(f)}})
STAB = pd.DataFrame(rows)
PIV = (STAB.groupby('cấu hình')
            .agg(rmse_mean_qua_seed=('cv_rmse_mean', 'mean'), rmse_std_qua_seed=('cv_rmse_mean', 'std'),
                 rmse_max_qua_seed=('cv_rmse_mean', 'max'), std_giua_cac_fold=('cv_rmse_std', 'mean'))
            .sort_values('rmse_mean_qua_seed'))
print(f'Kiểm tra ổn định: top-{TOP_K_STABLE} cấu hình x {len(STABLE_SEEDS)} seed = {len(STAB)} lần chạy CV')
print(PIV.round(5).to_string())
BEST_TAG = PIV.index[0]
_st, _tr = BEST_TAG.split('#')
_row = TRIALS[(TRIALS.stage == _st) & (TRIALS.trial == int(_tr))].iloc[0]
HP = post_process_hp({**cast_params({c: _row[c] for c in PARAM_COLS if pd.notna(_row[c])}), **FIXED_HP})
print(f'\n>>> CHỐT cấu hình: {BEST_TAG} '
      f'(CV RMSE trung bình qua {len(STABLE_SEEDS)} seed = {PIV.iloc[0]["rmse_mean_qua_seed"]:.4f} kWh)')
print('>>> HP cuối cùng:'); print(json.dumps({k: (round(v, 6) if isinstance(v, float) else v) for k, v in HP.items()},
                                             indent=2, ensure_ascii=False, default=str))
STAB.to_csv(f'{OUT_DIR}/{MODEL_TAG}_stability_check.csv', index=False)

## 7. Chẩn đoán quá khớp & learning curve

In [ ]:

# ===== CELL 10: Chẩn đoán quá khớp — so sánh lỗi trên Train / CV / Test của HP đã chốt =====
_m_dev = build_and_fit(X_FIT.iloc[DEV], Y_ALL[DEV], HP)
_rm_dev = np.sqrt(mean_squared_error(Y_ALL[DEV], predict_kwh(_m_dev, X_FIT.iloc[DEV])))
_f = cv_rmse_folds(build_and_fit, HP)
_m_te = np.sqrt(mean_squared_error(Y_ALL[TEST], predict_kwh(_m_dev, X_FIT.iloc[TEST])))
GAP = pd.DataFrame([{'In-sample RMSE (Train+Val, đã thấy)': _rm_dev,
                     'CV RMSE (5 fold, out-of-fold)': _f.mean(),
                     'Test RMSE (chưa từng thấy)': _m_te,
                     'Khoảng cách CV - in-sample': _f.mean() - _rm_dev,
                     'Khoảng cách Test - CV': _m_te - _f.mean()}], index=[MODEL_NAME]).T
print('Chẩn đoán quá khớp (kWh):'); print(GAP.round(4).to_string())
print('\nRMSE từng fold của HP đã chốt:', np.round(_f, 4).tolist(), '| std =', round(_f.std(), 4))

# Đường cong học theo lượng dữ liệu: fine-tuning còn ý nghĩa không nếu có thêm dữ liệu?
FRACS = [0.25, 0.5, 0.75, 1.0]
lc = []
for fr in FRACS:
    s = []
    for tr, va in FOLDS:
        k = max(200, int(len(tr) * fr)); tr2 = tr[-k:]            # giữ phần GẦN NHẤT (hợp với time-series)
        m = build_and_fit(X_FIT.iloc[tr2], Y_ALL[tr2], HP)
        s.append(np.sqrt(mean_squared_error(Y_ALL[va], predict_kwh(m, X_FIT.iloc[va]))))
    lc.append({'tỷ_lệ_train': fr, 'cv_rmse': float(np.mean(s))})
LC = pd.DataFrame(lc).set_index('tỷ_lệ_train')
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(GAP.index[:3], GAP[MODEL_NAME].values[:3], color=['#4C78A8', '#F58518', '#54A24B'])
ax[0].set(ylabel='RMSE (kWh)', title=f'{MODEL_NAME}: in-sample vs CV vs Test'); ax[0].tick_params(axis='x', rotation=18, labelsize=7)
ax[1].plot(LC.index, LC['cv_rmse'], 'o-', color='#4C78A8')
ax[1].set(xlabel='Phần dữ liệu train gần nhất được dùng', ylabel='CV RMSE (kWh)',
          title=f'{MODEL_NAME}: learning curve với HP đã chốt')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{MODEL_TAG}_diagnostics.png', dpi=130, bbox_inches='tight'); plt.show()
print('\nLearning curve:'); print(LC.round(4).to_string())

## 8. Benchmark cuối trên Test + xuất siêu tham số tốt nhất

In [ ]:

# ===== CELL 11: Benchmark cuối + XUẤT siêu tham số tốt nhất (đầu vào cho bài báo) =====
RES = run_benchmark(MODEL_NAME, build_and_fit, HP, X=X_FIT)
summarize(RES)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, m in zip(ax, ['RMSE', 'WAPE_%', 'R2']):
    v = RES['cv'][m]; a.bar(v.index, v.values, color='#4C78A8')
    a.axhline(v.mean(), color='k', ls='--', lw=1, label=f'mean = {v.mean():.3f}')
    a.set(title=f'{m} theo fold ({MODEL_NAME}, HP đã fine-tune)', xlabel='fold'); a.set_xticks(v.index); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{MODEL_TAG}_cv_folds_tuned.png', dpi=130, bbox_inches='tight'); plt.show()

BEST = {'rq': 'RQ1', 'model': MODEL_NAME, 'model_tag': MODEL_TAG, 'target': TARGET,
        'features': FEATURES, 'n_features': len(FEATURES),
        'split': {'TRAIN_END': TRAIN_END, 'VAL_END': VAL_END, 'TEST_END': TEST_END,
                  'n_train': int((df['set'] == 'Train').sum()), 'n_val': int((df['set'] == 'Val').sum()),
                  'n_test': int(len(TEST)), 'n_folds': N_FOLDS, 'fold_months': FOLD_MONTHS},
        'tuning': {'library': 'optuna ' + optuna.__version__, 'sampler': 'TPE (multivariate, grouped)',
                   'objective': 'mean CV RMSE over 5 expanding-window folds',
                   'n_trials_stage1': int(N_TRIALS_S1), 'n_trials_stage2': int(N_TRIALS_S2),
                   'n_trials_total': int(len(TRIALS)), 'top_k_stability': int(TOP_K_STABLE),
                   'stability_seeds': STABLE_SEEDS, 'chosen_config': BEST_TAG},
        'best_hyperparameters': {k: (float(v) if isinstance(v, (np.floating, float)) else
                                     int(v) if isinstance(v, (np.integer,)) else v) for k, v in HP.items()},
        'cv_rmse_best': float(TRIALS['cv_rmse'].min()),
        'cv_rmse_chosen_mean_over_seeds': float(PIV.iloc[0]['rmse_mean_qua_seed']),
        'cv': {k: float(v) for k, v in RES['cv'][METRICS].mean().items()},
        'cv_std': {k: float(v) for k, v in RES['cv'][METRICS].std().items()},
        'test': {k: float(v) for k, v in RES['test_all'].items()},
        'overfit_diagnostics': {k: float(v) for k, v in GAP[MODEL_NAME].items()},
        'learning_curve': {str(k): float(v) for k, v in LC['cv_rmse'].items()}}
with open(f'{OUT_DIR}/{MODEL_TAG}_best_hyperparameters.json', 'w', encoding='utf-8') as f:
    json.dump(BEST, f, indent=2, ensure_ascii=False, default=str)
RES['cv'].round(5).to_csv(f'{OUT_DIR}/{MODEL_TAG}_cv_folds_tuned.csv')
RES['test_all'].round(5).to_frame().T.to_csv(f'{OUT_DIR}/{MODEL_TAG}_test_tuned.csv', index=False)
RES['pred'].to_csv(f'{OUT_DIR}/{MODEL_TAG}_test_predictions.csv', index=False)
print('\nĐã lưu:')
for f_ in sorted(os.listdir(OUT_DIR)):
    if f_.startswith(MODEL_TAG): print('  ', f_)
print('\n================ TÓM TẮT CHO BÀI BÁO ================')
print(f'Model          : {MODEL_NAME}')
print(f'Best HP        : {BEST["best_hyperparameters"]}')
print(f'CV RMSE        : {BEST["cv"]["RMSE"]:.4f} ± {BEST["cv_std"]["RMSE"]:.4f} kWh')
print(f'CV MAE / R2    : {BEST["cv"]["MAE"]:.4f} kWh / {BEST["cv"]["R2"]:.4f}')
print(f'Test RMSE/MAE/R2: {BEST["test"]["RMSE"]:.4f} / {BEST["test"]["MAE"]:.4f} / {BEST["test"]["R2"]:.4f}')
print(f'Test WAPE      : {BEST["test"]["WAPE_%"]:.2f}%')
print('=====================================================')

## 9. Fine-tuning đem lại bao nhiêu so với cấu hình mặc định?

In [ ]:

# ===== CELL 12: So sánh HP mặc định / HP notebook gốc / HP sau fine-tuning =====
# Mục đích: định lượng fine-tuning đem lại bao nhiêu, để viết vào bài báo.
HP_DEFAULT = post_process_hp({**FIXED_HP})                       # LGBMRegressor mặc định
CMP = {}
for nm, hp in [('Mặc định (chưa tune)', HP_DEFAULT), ('Sau fine-tuning (notebook này)', HP)]:
    f = cv_rmse_folds(build_and_fit, hp)
    m = build_and_fit(X_FIT.iloc[DEV], Y_ALL[DEV], hp)
    pt = predict_kwh(m, X_FIT.iloc[TEST])
    CMP[nm] = {'CV RMSE': f.mean(), 'CV RMSE std': f.std(), **{f'Test {k}': v for k, v in reg_metrics(Y_ALL[TEST], pt).items() if k in METRICS}}
CMP = pd.DataFrame(CMP).T
CMP['Giảm CV RMSE so với mặc định %'] = 100 * (CMP.loc['Mặc định (chưa tune)', 'CV RMSE'] - CMP['CV RMSE']) / CMP.loc['Mặc định (chưa tune)', 'CV RMSE']
print('Fine-tuning đem lại bao nhiêu?'); print(CMP.round(4).to_string())
CMP.round(5).to_csv(f'{OUT_DIR}/{MODEL_TAG}_gain_vs_default.csv')

## 10. Biểu đồ cho bài báo (vẽ bằng seaborn/matplotlib ngay trong notebook này)
Các cell dưới đây đọc trực tiếp `df`, `HP`, `build_and_fit` của notebook và ghi PNG/CSV vào `OUT_DIR/paper_figures`. Không đọc bảng nào từ ngoài.

In [ ]:
# ===== PAPER FIGURES: style chung (seaborn + matplotlib), giữ nhất quán giữa các notebook =====
import seaborn as sns
OKABE = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'red': '#D55E00', 'purple': '#CC79A7', 'sky': '#56B4E9', 'yellow': '#F0E442', 'grey': '#7F7F7F', 'black': '#222222'}
sns.set_theme(context='paper', style='ticks', font='DejaVu Sans', rc={
    'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6.5, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'lines.linewidth': 1.2, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'figure.dpi': 110, 'savefig.dpi': 300})
FIG_DIR = os.path.join(OUT_DIR, 'paper_figures'); os.makedirs(FIG_DIR, exist_ok=True)
def panel_tag(ax, s):                                  # nhãn (a), (b), ... ở góc trên trái, như bài báo
    ax.text(-0.02, 1.08, s, transform=ax.transAxes, fontsize=8, fontweight='bold', va='bottom', ha='right')
def save_paper_fig(fig, name):
    p = os.path.join(FIG_DIR, name + '.png'); fig.savefig(p, bbox_inches='tight', pad_inches=0.03); print('Đã lưu', p)

In [ ]:
# ===== Fig. 3 của bài báo: độ quan trọng của 5 nhóm yếu tố (RQ1), tính ngay trong notebook này =====
# HP đã chốt (HP) -> fit trên Train+Val (DEV) -> chấm trên Test. Ba cách đo: hoán vị nhóm (20 lần), bỏ nhóm rồi fit lại, chỉ giữ nhóm rồi fit lại.
XF = globals().get('X_FIT', X_ALL)
GROUP_COLS = {g: list(c) for g, c in FACTOR_GROUPS.items()}
if '_hour_for_spline' in XF.columns: GROUP_COLS['hour'].append('_hour_for_spline')     # Linear Regression: cùng thông tin giờ, biểu diễn bằng spline
def set_const(X, cols):
    X = X.copy()
    for c in cols:
        v = X[c].iloc[0]
        X[c] = pd.Categorical([v] * len(X), categories=X[c].cat.categories) if isinstance(X[c].dtype, pd.CategoricalDtype) else v
    return X
def r2_of(m, X, y): return r2_score(y, predict_kwh(m, X))
Xd, Xt, yd, yt = XF.iloc[DEV], XF.iloc[TEST], Y_ALL[DEV], Y_ALL[TEST]
m_full = build_and_fit(Xd, yd, HP); R2_FULL = r2_of(m_full, Xt, yt)
rng = np.random.default_rng(0); rows = []
for g, cols in GROUP_COLS.items():
    drops = []
    for _ in range(N_PERM_REPEATS):
        idx = rng.permutation(len(Xt)); Z = Xt.copy()
        for c in cols:
            v = Xt[c].to_numpy()[idx]; Z[c] = pd.Categorical(v, categories=Xt[c].cat.categories) if isinstance(Xt[c].dtype, pd.CategoricalDtype) else v
        drops.append(R2_FULL - r2_of(m_full, Z, yt))
    Xdrop = set_const(XF, cols); r2_drop = r2_of(build_and_fit(Xdrop.iloc[DEV], yd, HP), Xdrop.iloc[TEST], yt)
    Xonly = set_const(XF, [c for g2, c2 in GROUP_COLS.items() if g2 != g for c in c2]); r2_only = r2_of(build_and_fit(Xonly.iloc[DEV], yd, HP), Xonly.iloc[TEST], yt)
    rows.append(dict(model=MODEL_NAME, group=g, perm_R2_loss=np.mean(drops), perm_R2_sd=np.std(drops), R2_drop=r2_drop, R2_only=r2_only, R2_full=R2_FULL))
IMPORT = pd.DataFrame(rows); IMPORT.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_rq1_group_importance.csv'), index=False)
print(IMPORT.round(4).to_string(index=False))

NAME = {'location': 'Location', 'hour': 'Time of day', 'month': 'Month, season', 'day_of_week': 'Day of week', 'is_holiday': 'Holiday'}
D = IMPORT.assign(Factor=IMPORT.group.map(NAME)).sort_values('perm_R2_loss', ascending=False); order = D.Factor.tolist()
fig, ax = plt.subplots(1, 3, figsize=(4.8, 1.55), sharey=True, gridspec_kw={'wspace': 0.1})
for a, col, ttl, c in zip(ax, ['perm_R2_loss', 'R2_drop', 'R2_only'], ['Permuted', 'Factor removed', 'Factor only'], [OKABE['blue'], OKABE['orange'], OKABE['green']]):
    sns.barplot(data=D, y='Factor', x=col, order=order, color=c, ax=a, width=0.62)
    if col == 'perm_R2_loss': a.errorbar(D.set_index('Factor').loc[order, col], range(len(order)), xerr=D.set_index('Factor').loc[order, 'perm_R2_sd'], fmt='none', ecolor='k', elinewidth=0.6, capsize=1.5)
    else: a.axvline(R2_FULL, color='k', lw=0.7, ls='--'); a.axvline(0, color='k', lw=0.5)
    for i, v in enumerate(D.set_index('Factor').loc[order, col]): a.text(0.99, i, f'{v:.3f}', va='center', ha='right', fontsize=5.8, transform=a.get_yaxis_transform())
    a.set_title(ttl); a.set_ylabel(''); a.set_xlabel('Loss of $R^2$' if col == 'perm_R2_loss' else 'Test $R^2$')
ax[0].set_xlim(0, 0.3); ax[0].set_xticks([0, 0.1, 0.2]); ax[1].set_xlim(0, 0.25); ax[2].set_xlim(-0.02, 0.25); [a.set_xticks([0, 0.1, 0.2]) for a in ax[1:]]
for a, t in zip(ax, 'abc'): panel_tag(a, f'({t})')
save_paper_fig(fig, f'fig3_rq1_importance_{MODEL_TAG}'); plt.show()

## Kết luận của notebook fine-tuning này

1. **Siêu tham số đã chốt** nằm trong `lightgbm_best_hyperparameters.json`, khoá `best_hyperparameters`.
   Đây là bộ sẽ được đưa vào bài báo.
2. **Cấu hình được chọn không phải trial có CV RMSE thấp nhất một cách tuyệt đối**, mà là cấu hình
   có CV RMSE **trung bình qua nhiều seed** tốt nhất trong top-K (cell 9). Giá trị thấp nhất của một lần
   chạy là giá trị nhỏ nhất trong hàng chục lượt thử nên nó lạc quan; trung bình qua seed đáng tin hơn.
3. **Cell 10** cho biết mô hình có quá khớp hay không: nếu `Test − CV` lớn hơn `std` giữa các fold
   đáng kể thì phần chênh đó đến từ **dịch chuyển phân phối giữa các giai đoạn** (Test rơi vào
   giai đoạn COVID và có khoảng trống dữ liệu dài), **không** phải do tuning quá tay.
4. **Cell 12** định lượng fine-tuning đem lại bao nhiêu so với cấu hình mặc định — con số này được
   báo cáo trong bài báo để cho thấy phần cải thiện là nhờ tuning, không phải nhờ đổi dữ liệu.

**Giới hạn:** notebook này chỉ tối ưu **RMSE của kWh/phiên** trên 5 fold đã định. Nó **không**
thay đổi tập feature, không thay đổi mốc chia tập, và **không** tính lại xếp hạng 5 nhóm yếu tố
của RQ1 — phần đó vẫn thuộc notebook RQ1 gốc, chỉ cần nạp bộ HP mới ở đây vào.